# N6 N3+N4 YOLO11l Segmentation Ensemble

Evaluates a mask-aware ensemble between the N3 EO+IR YOLO11l generalist and the N4 IR-only YOLO11l specialist. This notebook does not train. It extracts `best.pt` from `full_n3_results.zip` and `full_n4_results.zip`, evaluates the ensemble on IR-only and EO+IR validation sets, prints deltas against N3/N4, and packages the outputs.


In [ ]:
from pathlib import Path

GITHUB_REPO = "https://github.com/AyushPanchal/domain-adaptation-segmentation.git"
WORK_DIR = Path("/kaggle/working")
REPO_DIR = WORK_DIR / "domain-adaptation-segmentation"
GENERATED_ROOT = WORK_DIR / "generated" / "n6_n3_n4_ensemble"
WEIGHTS_DIR = WORK_DIR / "weights" / "n6_n3_n4_ensemble"
OUTPUT_ROOT = WORK_DIR / "runs" / "n6_n3_n4_ensemble"
EVAL_IR_YAML = "data/manifests/dataset_yamls/kaggle_n6_eval_ir.yaml"
EVAL_EO_IR_YAML = "data/manifests/dataset_yamls/kaggle_n6_eval_eo_ir.yaml"

DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/ayushbpanchal/indraeye-seg"),
    Path("/kaggle/input/indraeye-seg"),
]
DATASET_ROOT = next((path for path in DATASET_ROOT_CANDIDATES if path.exists()), DATASET_ROOT_CANDIDATES[0])

YOLO_EVAL_DEVICE = "0"
YOLO_BATCH = "8"
YOLO_WORKERS = "2"
IMG_SIZE = "640"

N3_BASELINE = {
    "eval_ir": {"mask_mAP50": 0.6968, "mask_mAP50_95": 0.4555, "box_mAP50": 0.7167, "box_mAP50_95": 0.6099},
    "eval_eo_ir": {"mask_mAP50": 0.5823, "mask_mAP50_95": 0.3308, "box_mAP50": 0.6192, "box_mAP50_95": 0.5025},
}
N4_BASELINE = {
    "eval_ir": {"mask_mAP50": 0.6555, "mask_mAP50_95": 0.4438, "box_mAP50": 0.7056, "box_mAP50_95": 0.5943},
    "eval_eo_ir": {"mask_mAP50": 0.3448, "mask_mAP50_95": 0.1984, "box_mAP50": 0.3768, "box_mAP50_95": 0.3011},
}

print("DATASET_ROOT:", DATASET_ROOT)
print("OUTPUT_ROOT:", OUTPUT_ROOT)


## Helpers

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile
from datetime import datetime

def stage(title):
    print("\n" + "=" * 92)
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {title}")
    print("=" * 92, flush=True)

def run_live(cmd, cwd=None, env=None):
    print("\nRUN:", " ".join(str(part) for part in cmd), flush=True)
    start = time.time()
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="")
    rc = process.wait()
    print(f"\nReturn code: {rc}; elapsed: {time.time() - start:.1f}s", flush=True)
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

def image_files(path):
    return sorted([p for p in Path(path).iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}])

def count_files(path, suffixes):
    return sum(1 for p in Path(path).iterdir() if p.suffix.lower() in suffixes)

def reset_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def copy_split_pairs(name, sources):
    out_images = GENERATED_ROOT / "images" / name
    out_labels = GENERATED_ROOT / "labels" / name
    reset_dir(out_images)
    reset_dir(out_labels)
    total = 0
    for prefix, image_dir, label_dir in sources:
        for img in image_files(image_dir):
            label = label_dir / f"{img.stem}.txt"
            if not label.exists():
                continue
            dst_stem = f"{prefix}_{img.stem}"
            shutil.copy2(img, out_images / f"{dst_stem}{img.suffix.lower()}")
            shutil.copy2(label, out_labels / f"{dst_stem}.txt")
            total += 1
    print(f"{name}: {total} image/label pairs")

def class_block():
    return """nc: 12
names:
  0: Bicycle
  1: Bus
  2: Car
  3: Cargo trike
  4: Ignore
  5: Motorcycle
  6: Person
  7: Rickshaw
  8: Small truck
  9: Tractor
  10: Truck
  11: Van
"""

def find_zip(filename):
    candidates = []
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            candidates.extend(root.rglob(filename))
    if not candidates:
        raise FileNotFoundError(f"Could not find {filename}. Add it as a Kaggle input dataset or place it in /kaggle/working.")
    return sorted(candidates, key=lambda path: len(str(path)))[0]

def extract_best_pt(zip_path, token, output_name):
    output_path = WEIGHTS_DIR / output_name
    WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        members = [name for name in zf.namelist() if name.endswith("/weights/best.pt") and token in name]
        if not members:
            members = [name for name in zf.namelist() if name.endswith("/weights/best.pt")]
        if not members:
            raise FileNotFoundError(f"No best.pt found in {zip_path}")
        output_path.write_bytes(zf.read(members[0]))
    print(output_path, output_path.stat().st_size)
    return output_path


## Stage 1 - Clone Repo And Install

In [ ]:
stage("Stage 1 - Clone repository and install dependencies")
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    run_live(["git", "pull"], cwd=REPO_DIR)
elif REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
    run_live(["git", "clone", GITHUB_REPO, str(REPO_DIR)])
else:
    run_live(["git", "clone", GITHUB_REPO, str(REPO_DIR)])
run_live(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR)
run_live([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], cwd=REPO_DIR)


## Stage 2 - Prepare Evaluation Data And Weights

In [ ]:
stage("Stage 2 - Prepare eval folders, YAMLs, and best.pt files")
required_dirs = [
    DATASET_ROOT / "eo/images/val",
    DATASET_ROOT / "eo/labels/val",
    DATASET_ROOT / "ir/images/val",
    DATASET_ROOT / "ir/labels/val",
]
for path in required_dirs:
    print(path, "OK" if path.exists() else "MISSING")
    if not path.exists():
        raise FileNotFoundError(path)

copy_split_pairs("eval_ir", [("ir", DATASET_ROOT / "ir/images/val", DATASET_ROOT / "ir/labels/val")])
copy_split_pairs("eval_eo_ir", [
    ("eo", DATASET_ROOT / "eo/images/val", DATASET_ROOT / "eo/labels/val"),
    ("ir", DATASET_ROOT / "ir/images/val", DATASET_ROOT / "ir/labels/val"),
])

for yaml_rel, split in [(EVAL_IR_YAML, "eval_ir"), (EVAL_EO_IR_YAML, "eval_eo_ir")]:
    yaml_path = REPO_DIR / yaml_rel
    yaml_path.parent.mkdir(parents=True, exist_ok=True)
    yaml_path.write_text(f"""path: {GENERATED_ROOT.as_posix()}
train: images/{split}
val: images/{split}

{class_block()}""", encoding="utf-8")
    print(f"\n--- {yaml_path} ---")
    print(yaml_path.read_text())

n3_zip = find_zip("full_n3_results.zip")
n4_zip = find_zip("full_n4_results.zip")
print("N3 zip:", n3_zip)
print("N4 zip:", n4_zip)
n3_best = extract_best_pt(n3_zip, "N3", "n3_best.pt")
n4_best = extract_best_pt(n4_zip, "N4", "n4_best.pt")


## Stage 3 - Run Ensemble Evaluation

In [ ]:
stage("Stage 3 - Evaluate N3+N4 ensemble")
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO_DIR / "src")

eval_root = OUTPUT_ROOT / "evaluations"
for eval_name, eval_yaml in [("eval_ir", EVAL_IR_YAML), ("eval_eo_ir", EVAL_EO_IR_YAML)]:
    cmd = [
        sys.executable, "-m", "domain_adaptation_segmentation.training.evaluate_segmentation_ensemble",
        "--weights", str(n3_best), str(n4_best),
        "--data", eval_yaml,
        "--output-root", str(eval_root),
        "--name", eval_name,
        "--device", YOLO_EVAL_DEVICE,
        "--imgsz", IMG_SIZE,
        "--batch", YOLO_BATCH,
        "--workers", YOLO_WORKERS,
    ]
    run_live(cmd, cwd=REPO_DIR, env=env)


## Stage 4 - Print Results

In [ ]:
stage("Stage 4 - Results and deltas")
import pandas as pd

rows = []
for eval_name in ["eval_ir", "eval_eo_ir"]:
    metrics_path = OUTPUT_ROOT / "evaluations" / eval_name / "metrics.json"
    payload = json.loads(metrics_path.read_text(encoding="utf-8"))
    metrics = payload["metrics"]
    row = {
        "eval": eval_name,
        "box_mAP50": metrics.get("metrics/mAP50(B)"),
        "box_mAP50_95": metrics.get("metrics/mAP50-95(B)"),
        "mask_mAP50": metrics.get("metrics/mAP50(M)"),
        "mask_mAP50_95": metrics.get("metrics/mAP50-95(M)"),
    }
    for metric in ["mask_mAP50", "mask_mAP50_95", "box_mAP50", "box_mAP50_95"]:
        row[f"delta_vs_N3_{metric}"] = row[metric] - N3_BASELINE[eval_name][metric]
        row[f"delta_vs_N4_{metric}"] = row[metric] - N4_BASELINE[eval_name][metric]
    rows.append(row)

df = pd.DataFrame(rows)
display(df)

summary_json = OUTPUT_ROOT / "n6_n3_n4_summary.json"
summary_csv = OUTPUT_ROOT / "n6_n3_n4_summary.csv"
summary_json.write_text(json.dumps(rows, indent=2), encoding="utf-8")
df.to_csv(summary_csv, index=False)
print("Wrote", summary_json)
print("Wrote", summary_csv)


## Stage 5 - Package Results

In [ ]:
stage("Stage 5 - Package outputs")
from IPython.display import FileLink, display

bundle_dir = WORK_DIR / "n6_n3_n4_ensemble_artifacts"
if bundle_dir.exists():
    shutil.rmtree(bundle_dir)
bundle_dir.mkdir(parents=True, exist_ok=True)

if OUTPUT_ROOT.exists():
    shutil.copytree(OUTPUT_ROOT, bundle_dir / OUTPUT_ROOT.name)
for yaml_rel in [EVAL_IR_YAML, EVAL_EO_IR_YAML]:
    dst = bundle_dir / "yamls" / Path(yaml_rel).name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(REPO_DIR / yaml_rel, dst)

(bundle_dir / "ensemble_inputs.json").write_text(json.dumps({
    "n3_zip": str(n3_zip),
    "n4_zip": str(n4_zip),
    "n3_best": str(n3_best),
    "n4_best": str(n4_best),
    "dataset_root": str(DATASET_ROOT),
}, indent=2), encoding="utf-8")

archive_base = WORK_DIR / "n6_n3_n4_ensemble_results"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=bundle_dir)
print("Result zip:", archive_path)
display(FileLink(archive_path))
